## import packages

In [20]:
from pyflamegpu import *
import pyflamegpu.codegen
import sys
import math

In [21]:
import os

# ipynb 相比py文件，多了两行初始化代码
# change the working directory to the project root
project_root = r"D:\programming\python_script\socio-physical system for shelter"
os.chdir(project_root)




## Define the FLAME GPU model name

In [22]:
# Define the FLAME GPU model: 这个可以在后续的可视化窗口改名字
model = pyflamegpu.ModelDescription("Social_physical_shelter_Opt")

## host function for env

### for familirity map

### for graph

## messages

In [23]:
# 可以获取一定距离内的消息
message = model.newMessageSpatial3D("location")
# Configure the message list
message.setMin(0, 0,0)
message.setMax(20, 20,20)
message.setRadius(2)
# Add extra variables to the message
# X Y (Z) are implicit for spatial messages
message.newVariableID("id")

In [24]:
stairwell_message = model.newMessageSpatial3D("location_stairwell")
stairwell_message.setMin(0, 0,0)
stairwell_message.setMax(500, 500, 500)
stairwell_message.setRadius(200)
stairwell_message.newVariableID("id")
stairwell_message.newVariableFloat("class")

## agents set variables

### student agents

In [25]:
# Assign the agent some variables (ID is implicit to agents, so we don't define it ourselves)
student_agent = model.newAgent("student_agent")
student_agent.newVariableFloat("x")
student_agent.newVariableFloat("y")
student_agent.newVariableInt("building_id")
student_agent.newVariableInt("point_id")
student_agent.newVariableFloat("z")
student_agent.newVariableFloat("drift", 0)
#set the states for student agents
student_agent.newState("not evacuate")
student_agent.newState("focused")
student_agent.newState("building evacuate")
student_agent.newState("stairwell evacuate")
student_agent.newState("neighborhood evacuate")



### stairwell agents

In [26]:
stairwell_agent = model.newAgent("stairwell_agent")
stairwell_agent.newVariableFloat("x")
stairwell_agent.newVariableFloat("y")
stairwell_agent.newVariableInt("stairwell_id")
stairwell_agent.newVariableFloat("z")

## environment

In [27]:
# Fetch the model's environment
env = model.Environment()

### familiarity map

### road planning graph

## agent functions

In [28]:
#根据building_id前往随机一个building的stairwell。获取student agent的building_id和楼层floors，来判断前往的特定stairwell的location，最近的或者最熟悉的stairwell任意选一个。
@pyflamegpu.agent_function
def set_target_stairwell(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageNone):
    # 获取当前agent的building_id
    building_id = pyflamegpu.getVariableInt("building_id")
    
    # 获取当前agent的楼层floors
    floors = pyflamegpu.getVariableFloat("z")

    # 根据building_id和floors判断前往的特定stairwell的location
    # 这里需要根据实际情况进行判断，例如根据building_id和floors判断前往的特定stairwell的location
    # 这里暂时返回一个固定的location


    return pyflamegpu.ALIVE
    
    

## function write in

In [29]:
# translate the agent functions from Python to C++
#output_func_translated = pyflamegpu.codegen.translate(output_message)
#input_func_translated = pyflamegpu.codegen.translate(input_message)

#ExampleFn_translated = pyflamegpu.codegen.translate(ExampleFn)
set_target_stairwell_translated = pyflamegpu.codegen.translate(set_target_stairwell)
#ExampleFn_fn = agent.newRTCFunction("ExampleFn",ExampleFn_translated)
set_target_stairwell_fn = student_agent.newRTCFunction("set_target_stairwell",set_target_stairwell_translated)


In [30]:
model.addExecutionRoot(set_target_stairwell_fn)
model.generateLayers()

## model simulation write in

In [31]:
# Specify the desired StepLoggingConfig
step_log_cfg = pyflamegpu.StepLoggingConfig(model)
# Log every step
step_log_cfg.setFrequency(1)
# Include the mean of the "point" agent population's variable 'drift'
step_log_cfg.agent("student_agent").logMeanFloat("z")

# Create and init the simulation
cuda_model = pyflamegpu.CUDASimulation(model)

FLAMEGPURuntimeException: (InvalidAgentState) D:\a\FLAMEGPU2\FLAMEGPU2\src\flamegpu\simulation\LoggingConfig.cu(34): State 'default' was not found within agent 'student_agent' in the model description, in LoggingConfig::agent()


## agent initialization

In [ ]:
# 导入flamegpu_init_code.py中的初始化函数

sys.path.append('data/output')
from flamegpu_init_code import initialize_student_agent_population

# 添加学生代理类型
# 基于data/output/flamegpu_init_code.py的学生代理初始化

# 初始化学生代理种群
initialize_student_agent_population(model, cuda_model)

# Specify the desired StepLoggingConfig
step_log_cfg = pyflamegpu.StepLoggingConfig(model)
# Log every step
step_log_cfg.setFrequency(1) 
# Include the mean of the "point" agent population's variable 'drift'
step_log_cfg.agent("student_agent").logMeanFloat("drift")
step_log_cfg.agent("student_agent").logMeanFloat("x")
step_log_cfg.agent("student_agent").logMeanFloat("y")


cuda_model.initialise(sys.argv)


# Attach the logging config
cuda_model.setStepLog(step_log_cfg) 

## visualization part

In [ ]:
# Only run this block if pyflamegpu was built with visualisation support
if pyflamegpu.VISUALISATION:
    # Create visualisation
    m_vis = cuda_model.getVisualisation()
    # Set the initial camera location and speed

    m_vis.setInitialCameraTarget(270, 205, 0)
    m_vis.setInitialCameraLocation(240, 100, 100)
    m_vis.setCameraSpeed(0.01)
    m_vis.setSimulationSpeed(25)
    # Add "point" agents to the visualisation

    
    # Add "student_agent" agents to the visualisation
    student_agt = m_vis.addAgent("student_agent")
    student_agt.setModel(pyflamegpu.ICOSPHERE);
    student_agt.setModelScale(1/1.0);
    # Mark the environment bounds.

    stairwell_agt = m_vis.addAgent("stairwell_agent")
    stairwell_agt.setModel(pyflamegpu.ICOSPHERE);
    stairwell_agt.setModelScale(1/0.5);
    stairwell_agt.setColor(pyflamegpu.RED);
     
    pen = m_vis.newPolylineSketch(1, 1, 1, 0.2)
    pen.addVertex(275, 637, 0) # 起始点
    pen.addVertex(69, 510, 0)
    pen.addVertex(0, 301, 0)
    pen.addVertex(1, 167, 0)
    pen.addVertex(29, 142, 0)
    pen.addVertex(57, 98, 0)
    pen.addVertex(118, 67, 0)
    pen.addVertex(109, 24, 0)
    pen.addVertex(287, 0, 0)
    pen.addVertex(286, 45, 0)
    pen.addVertex(405, 154, 0)
    pen.addVertex(435, 131, 0)
    pen.addVertex(436, 72, 0)
    pen.addVertex(467, 41, 0)
    pen.addVertex(501, 35, 0)
    pen.addVertex(543, 47, 0)
    pen.addVertex(275, 637, 0) # 闭合点
    # Open the visualiser window 
    m_vis.activate()


# Run the simulation
for i in range(100):
        cuda_model.step()

if pyflamegpu.VISUALISATION:
    # Keep the visualisation window active after the simulation has completed
    m_vis.join()


# python src/test_3d_with_function.py -s 10 --out-step step.json

In [ ]:
out_pop = pyflamegpu.AgentVector(model.Agent("student_agent"))
cuda_model.getPopulationData(out_pop)
for agent in out_pop:
    print("   %f"%(agent.getVariableFloat("building_id")))
    print("   %f"%(agent.getVariableFloat("z")))